In [30]:
# Install required libraries
!pip install --upgrade --quiet datasets[audio]

In [31]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
# Imports
import sys, os, json, random
import pandas as pd
from datasets import load_from_disk, Audio

sys.path.insert(0, '/content/drive/MyDrive/Target TTS Project/TargetTTS')

In [33]:
# Base Path
BASE_PATH = "/content/drive/MyDrive/Target TTS Project/TargetTTS"

# LibriSpeech
LIBRISPEECH_MANIFEST = f"{BASE_PATH}/data/librispeech/manifests/test-clean.jsonl"
ENG_OUTPUT_CSV       = f"{BASE_PATH}/data/mixture_recipes/librispeech_mixture_recipes.csv"
ENG_DIST_LOG         = f"{BASE_PATH}/data/mixture_recipes/librispeech_distribution_log.txt"

# WAXAL
WAXAL_DISK_PATH = f"{BASE_PATH}/data/waxal"
TWI_OUTPUT_CSV  = f"{BASE_PATH}/data/mixture_recipes/waxal_mixture_recipes.csv"
TWI_DIST_LOG    = f"{BASE_PATH}/data/mixture_recipes/waxal_distribution_log.txt"

# Configuration
SIR_LEVELS     = [-5, 0, 5]                       # in decibels
OVERLAP_RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5, 1.0]
NUM_MIXES      = 12500                            # number of mixes per dataset

In [34]:
# Metadata Loaders
def load_librispeech_metadata(manifest_path):
    """Load audio pool entries from a LibriSpeech JSONL manifest."""
    entries = []
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            entries.append({
                "dataset":    "librispeech",
                "filepath":   data["audio_path"],
                "speaker_id": str(data["speaker_id"]),
                "gender":     "Unknown",
                "transcript": data["transcript"],
                "duration_s": data["duration_s"],
            })
    print(f"Loaded {len(entries)} LibriSpeech entries.")
    return entries


def load_waxal_metadata(dataset_path, split="test"):
    """Load audio pool entries from the WAXAL HuggingFace dataset."""
    print(f"Loading WAXAL dataset from {dataset_path}...")
    ds = load_from_disk(dataset_path)[split].cast_column("audio", Audio(decode=False))

    entries = []
    for item in ds:
        audio_info = item["audio"]
        filepath = audio_info.get("path") or f"waxal_id_{item['id']}"
        entries.append({
            "dataset":    "waxal",
            "filepath":   filepath,
            "speaker_id": str(item["speaker_id"]),
            "gender":     item["gender"],
            "transcript": item["transcription"],
            "duration_s": 0.0,
        })
    print(f"Loaded {len(entries)} WAXAL entries from '{split}' split.")
    return entries

In [35]:
# Recipe Generator & Saver
def generate_mix_recipes(audio_pool, num_mixes=NUM_MIXES):
    """
    Randomly pair target and interferer utterances from different speakers,
    assigning random SIR levels and overlap ratios.
    """
    recipes = []
    for i in range(num_mixes):
        target = random.choice(audio_pool)
        interferer = random.choice(audio_pool)
        while interferer["speaker_id"] == target["speaker_id"]:
            interferer = random.choice(audio_pool)

        recipes.append({
            "mix_id":             f"mix_{target['dataset']}_{i:04d}",
            "dataset":            target["dataset"],
            "target_audio_path":  target["filepath"],
            "target_speaker_id":  target["speaker_id"],
            "target_gender":      target["gender"],
            "target_transcript":  target["transcript"],
            "noise_audio_path":   interferer["filepath"],
            "noise_speaker_id":   interferer["speaker_id"],
            "noise_gender":       interferer["gender"],
            "noise_transcript":   interferer["transcript"],
            "sir_level_db":       random.choice(SIR_LEVELS),
            "overlap_ratio":      random.choice(OVERLAP_RATIOS),
            "target_start_time":  0.0,
            "noise_start_times":  [],
        })
    return pd.DataFrame(recipes)


def save_recipes(df, output_csv, dist_log, label):
    """Save mix recipes CSV and distribution log, and print a summary."""
    dist = pd.crosstab(df['sir_level_db'], df['overlap_ratio'])
    print(f"\n=== {label} Mix Distribution ===")
    print(dist.to_string())

    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df.to_csv(output_csv, index=False)

    with open(dist_log, 'w') as f:
        f.write(f"{label} Mix Distribution\n")
        f.write(dist.to_string())

    print(f"\nSaved {len(df)} recipes → {output_csv}")
    print(f"Saved distribution log → {dist_log}")

In [36]:
# English (LibriSpeech Dataset)
libri_data   = load_librispeech_metadata(LIBRISPEECH_MANIFEST)
df_libri     = generate_mix_recipes(libri_data, num_mixes=NUM_MIXES)

Loaded 2620 LibriSpeech entries.


In [24]:
save_recipes(df_libri, ENG_OUTPUT_CSV, ENG_DIST_LOG, "English (LibriSpeech)")


=== English (LibriSpeech) Mix Distribution ===
overlap_ratio  0.1  0.2  0.3  0.4  0.5  1.0
sir_level_db                               
-5             700  723  669  682  704  701
 0             708  726  692  700  666  672
 5             686  655  737  707  704  668

Saved 12500 recipes → /content/drive/MyDrive/Target TTS Project/TargetTTS/mixes_metadata/librispeech_mixture_recipes.csv
Saved distribution log → /content/drive/MyDrive/Target TTS Project/TargetTTS/mixes_metadata/librispeech_distribution_log.txt


In [37]:
# Twi (WAXAL Dataset)
waxal_data = load_waxal_metadata(WAXAL_DISK_PATH, split="test")
df_waxal   = generate_mix_recipes(waxal_data, num_mixes=NUM_MIXES)

Loading WAXAL dataset from /content/drive/MyDrive/Target TTS Project/TargetTTS/data/waxal...


Loading dataset from disk:   0%|          | 0/109 [00:00<?, ?it/s]

Loaded 1522 WAXAL entries from 'test' split.


In [26]:
save_recipes(df_waxal, TWI_OUTPUT_CSV, TWI_DIST_LOG, "Twi (WAXAL)")


=== Twi (WAXAL) Mix Distribution ===
overlap_ratio  0.1  0.2  0.3  0.4  0.5  1.0
sir_level_db                               
-5             668  653  726  719  744  711
 0             689  680  707  716  714  699
 5             639  697  670  669  715  684

Saved 12500 recipes → /content/drive/MyDrive/Target TTS Project/TargetTTS/mixes_metadata/waxal_mixture_recipes.csv
Saved distribution log → /content/drive/MyDrive/Target TTS Project/TargetTTS/mixes_metadata/waxal_distribution_log.txt
